# Reuters topic classification with DSPy: results

Interactive view of the steps in the `0X_*.py` scripts. LM calls are cached by DSPy, so re-running cells is free once a script has run.

In [1]:
from collections import Counter

import dspy
import pandas as pd

from common import TOPICS, TopicOf, evaluator, setup

trainset, devset = setup()
print(f"train: {len(trainset)}  dev: {len(devset)}")
Counter(e.topic for e in devset)

train: 56  dev: 100


Counter({'crude': 15,
         'acq': 15,
         'earn': 15,
         'money-fx': 15,
         'trade': 15,
         'interest': 15,
         'grain': 10})

## Step 1: zero-shot baseline (`dspy.Predict`)

One prediction by hand, then the prompt DSPy built from the signature.

In [2]:
baseline = dspy.Predict(TopicOf)

pred = baseline(text=devset[0].text)
print(f"gold={devset[0].topic!r}")
pred

gold='crude'


Prediction(
    topic='crude'
)

In [3]:
dspy.inspect_history(n=1)





[2026-10-22T07:38:14.404063]

System message:

Your input fields are:
1. `text` (str):
Your output fields are:
1. `topic` (Literal['earn', 'acq', 'crude', 'money-fx', 'grain', 'trade', 'interest']):
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## text ## ]]
{text}

[[ ## topic ## ]]
{topic}        # note: the value you produce must exactly match (no extra characters) one of: earn; acq; crude; money-fx; grain; trade; interest

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Classify a 1987 Reuters newswire article into its topic.


User message:

[[ ## text ## ]]
SOUTHLAND &lt;SLC> UNIT RAISES CRUDE OIL PRICES
  Citgo Petroleum Corp, a subsidiary of
  Southland Corp, said it raised the contract price it will pay
  for all grades of crude oil by 50 cts a barrel, effective Oct
  16
      The increase brings Citgo's postings for the West Texas
  Intermediate and West Texas Sour grades to 19.00 dlrs

### Evaluation on the dev set

In [4]:
def to_frame(result) -> pd.DataFrame:
    """One row per dev example: headline, gold label, predicted label."""
    return pd.DataFrame(
        {
            "headline": ex.text.strip().split("\n")[0][:80],
            "gold": ex.topic,
            "pred": p.topic,
            "correct": bool(ok),
        }
        for ex, p, ok in result.results
    )


baseline_result = evaluator(devset, display_progress=False)(baseline)
df = to_frame(baseline_result)
print(f"accuracy: {baseline_result.score}%")

2026/10/22 07:38:14 INFO dspy.evaluate.evaluate: Average Metric: 95 / 100 (95.0%)


accuracy: 95.0%


Mistakes:

In [5]:
df[~df.correct]

,headline,gold,pred,correct
19,19-OCT-1987,earn,money-fx,False
44,BANK OF JAPAN DETERMINED TO KEEP EASY MONEY PO...,interest,money-fx,False
58,STOLTENBERG ASSUMES COOPERATION WILL CONTINUE,money-fx,interest,False
59,SAUDI RATES RISE AS BAHRAIN BANKS CAUGHT SHORT,interest,money-fx,False
69,LYNG DEFENDS U.S. EXPORT SUBSIDY PROGRAM,trade,grain,False


Confusion matrix (rows: gold, columns: predicted) and accuracy per topic:

In [6]:
pd.crosstab(df.gold, df.pred).reindex(index=TOPICS, columns=TOPICS, fill_value=0)

pred,earn,acq,crude,money-fx,grain,trade,interest
gold,,,,,,,
earn,14,0,0,1,0,0,0
acq,0,15,0,0,0,0,0
crude,0,0,15,0,0,0,0
money-fx,0,0,0,14,0,0,1
grain,0,0,0,0,10,0,0
trade,0,0,0,0,1,14,0
interest,0,0,0,2,0,0,13


In [7]:
df.groupby("gold").correct.mean().reindex(TOPICS).round(2)

gold
earn        0.93
acq         1.00
crude       1.00
money-fx    0.93
grain       1.00
trade       0.93
interest    0.87
Name: correct, dtype: float64